# DBSCAN Cluster CSV -> Neo4j Graph Database -> Qwen RAG

This notebook loads the `dbscan_Blacksburg_VA_clusters.csv` output, merges it with the older `hex_cells_from_DBScan_K1_R10.csv` geometry table when available, populates Neo4j, and prepares vector retrieval + Qwen/Ollama RAG.

**Main structure:**

- `dbscan_Blacksburg_VA_clusters.csv` is the source of truth for cluster labels.
- `hex_cells_from_DBScan_K1_R10.csv` is useful as a geometry lookup table.
- Neo4j stores each H3 cell as a node, each cluster as a node, and assignment relationships between them.

## 0. Install packages

Run this once. If packages are already installed, it is safe to rerun.

In [1]:
%pip install pandas numpy neo4j h3 folium sentence-transformers requests ollama

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Imports and file paths

Place these files in the same folder as this notebook:

- `dbscan_Blacksburg_VA_clusters.csv` — the cluster assignments
- `hex_cells_from_DBScan_K1_R10.csv` — optional, but recommended for geometry and centroids

Change input file names if different files are run

In [2]:
from pathlib import Path
import json
import math
import warnings
from collections import Counter

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ---- Input files ----
CLUSTER_CSV = Path("dbscan_Blacksburg_VA_clusters.csv")
HEX_CSV = Path("hex_cells_from_DBScan_K1_R10.csv")

# ---- Output folder ----
OUTPUT_DIR = Path("neo4j_updated_cluster_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ---- Neo4j run name ----
RUN_NAME = "DBSCAN_Blacksburg_VA_updated"
PLACE_NAME = "Blacksburg, VA"
ALGORITHM = "DBSCAN"

print("Notebook folder:", Path.cwd())
print("Cluster CSV exists:", CLUSTER_CSV.exists(), CLUSTER_CSV.resolve())
print("Hex geometry CSV exists:", HEX_CSV.exists(), HEX_CSV.resolve())

Notebook folder: c:\Users\tdkim\Downloads\ClusteringOutputRag-20260505T223622Z-3-001\ClusteringOutputRag
Cluster CSV exists: True C:\Users\tdkim\Downloads\ClusteringOutputRag-20260505T223622Z-3-001\ClusteringOutputRag\dbscan_Blacksburg_VA_clusters.csv
Hex geometry CSV exists: True C:\Users\tdkim\Downloads\ClusteringOutputRag-20260505T223622Z-3-001\ClusteringOutputRag\hex_cells_from_DBScan_K1_R10.csv


## 2. Load the cluster CSV

Expected columns:

- `region_id` — H3 cell identifier
- `cluster` — DBSCAN cluster label

DBSCAN convention: `cluster = -1` means noise/outlier. Non-negative labels (`0`, `1`, `2`, etc.) are actual clusters.

In [ ]:
if not CLUSTER_CSV.exists():
    raise FileNotFoundError(
        f"Could not find {CLUSTER_CSV.resolve()}. "
        "Put dbscan_Blacksburg_VA_clusters.csv in the same folder as this notebook."
    )

#reads in cluster data from csv
clusters_raw_df = pd.read_csv(CLUSTER_CSV)
clusters_raw_df.columns = [c.strip().lower() for c in clusters_raw_df.columns]

required_cols = {"region_id", "cluster"}
missing = required_cols - set(clusters_raw_df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}. Expected region_id and cluster.")

cluster_df = clusters_raw_df[["region_id", "cluster"]].copy()
cluster_df["region_id"] = cluster_df["region_id"].astype(str).str.strip()
cluster_df["cluster"] = cluster_df["cluster"].astype(int)

before = len(cluster_df)
cluster_df = cluster_df.drop_duplicates(subset=["region_id"], keep="last")
after = len(cluster_df)

print("Updated cluster CSV shape:", clusters_raw_df.shape)
print("Rows before duplicate cleanup:", before)
print("Rows after duplicate cleanup:", after)
print("Unique DBSCAN labels:", cluster_df["cluster"].nunique())
print("Noise/outlier cells:", int((cluster_df["cluster"] == -1).sum()))

cluster_counts = cluster_df["cluster"].value_counts().sort_index()
display(cluster_counts.rename("cell_count").to_frame())
display(cluster_df.head())

Updated cluster CSV shape: (3836, 2)
Rows before duplicate cleanup: 3836
Rows after duplicate cleanup: 3836
Unique DBSCAN labels: 38
Noise/outlier cells: 818


,cell_count
cluster,
-1,818
0,301
1,5
2,20
3,559
4,345
5,362
6,56
7,474


,region_id,cluster
0,8a2a8a892717fff,0
1,8a2a8ad4a0d7fff,1
2,8a2a8ad5931ffff,2
3,8a2a8ad4ad87fff,3
4,8a2a8ac66557fff,4


## 3. H3 functions

These are used to compute centroids and polygon boundaries from H3 `region_id` values when the older geometry CSV does not contain a given cell.

In [ ]:
import h3

#Grabs latitude and longitude coordinates based on region id (hex)
def get_h3_lat_lon(region_id: str):
    """Return (latitude, longitude) for an H3 cell."""
    region_id = str(region_id)
    if hasattr(h3, "cell_to_latlng"):
        lat, lon = h3.cell_to_latlng(region_id)
    else:
        lat, lon = h3.h3_to_geo(region_id)
    return float(lat), float(lon)

#Gets the geo_json information for a region id (hex)
def get_h3_geometry_json(region_id: str):
    """
    Return GeoJSON-style Polygon geometry JSON for an H3 cell.
    GeoJSON uses [longitude, latitude] coordinate order.
    """
    region_id = str(region_id)
    if hasattr(h3, "cell_to_boundary"):
        boundary = h3.cell_to_boundary(region_id)
    else:
        boundary = h3.h3_to_geo_boundary(region_id)

    # h3 returns boundary points as (lat, lon). GeoJSON needs [lon, lat].
    coords = [[float(lon), float(lat)] for lat, lon in boundary]

    # Close polygon ring.
    if coords and coords[0] != coords[-1]:
        coords.append(coords[0])

    geom = {"type": "Polygon", "coordinates": [coords]}
    return json.dumps(geom, separators=(",", ":"))

#chooses color for a given cluster
def cluster_color(cluster_id: int):
    """Simple repeating color palette. Noise/outliers are gray."""
    cluster_id = int(cluster_id)
    if cluster_id == -1:
        return "#808080"

    palette = [
        "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
        "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
        "#aec7e8", "#ffbb78", "#98df8a", "#ff9896", "#c5b0d5",
        "#c49c94", "#f7b6d2", "#c7c7c7", "#dbdb8d", "#9edae5"
    ]
    return palette[cluster_id % len(palette)]

print("H3 helpers ready.")

H3 helpers ready.


## 4. Build the full HexCell table

This cell creates the main table to load into Neo4j.

- The cluster CSV provides the **cluster labels**.
- The old hex CSV provides geometry, centroid, old feature ID, and old `rag_text` when available.
- Any missing geometry/centroids are computed directly from the H3 index.

In [5]:
# source of clustering assignments.
hex_df = cluster_df.rename(columns={"cluster": "cluster_id"}).copy()

# Optional old geometry table.
if HEX_CSV.exists():
    old_hex_df = pd.read_csv(HEX_CSV)
    old_hex_df.columns = [c.strip() for c in old_hex_df.columns]
    old_hex_df["region_id"] = old_hex_df["region_id"].astype(str).str.strip()

    keep_cols = [
        "region_id",
        "feature_id",
        "centroid_lat",
        "centroid_lon",
        "area_degrees_sq",
        "geometry_json"
    ]
    keep_cols = [c for c in keep_cols if c in old_hex_df.columns]

    hex_df = hex_df.merge(old_hex_df[keep_cols], on="region_id", how="left")
    print(f"Merged geometry/centroid fields from {HEX_CSV.name}.")
else:
    print(f"{HEX_CSV.name} not found. Will compute all centroids/geometries from H3 IDs.")

# Feature ID is an internal map/export ID; create one if missing.
if "feature_id" not in hex_df.columns:
    hex_df["feature_id"] = np.arange(len(hex_df))
else:
    # Keep old feature ID where available; fill missing with new sequence numbers.
    missing_feature = hex_df["feature_id"].isna()
    hex_df.loc[missing_feature, "feature_id"] = np.arange(missing_feature.sum()) + 10_000_000
    hex_df["feature_id"] = hex_df["feature_id"].astype(int)

# Color is based on the updated cluster label, not the old CSV.
hex_df["color"] = hex_df["cluster_id"].apply(cluster_color)

# Ensure centroid columns exist.
if "centroid_lat" not in hex_df.columns:
    hex_df["centroid_lat"] = np.nan
if "centroid_lon" not in hex_df.columns:
    hex_df["centroid_lon"] = np.nan

# Fill missing centroids from H3.
missing_centroids = hex_df["centroid_lat"].isna() | hex_df["centroid_lon"].isna()
print("Rows missing centroid before H3 fill:", int(missing_centroids.sum()))

for idx in hex_df[missing_centroids].index:
    lat, lon = get_h3_lat_lon(hex_df.at[idx, "region_id"])
    hex_df.at[idx, "centroid_lat"] = lat
    hex_df.at[idx, "centroid_lon"] = lon

# Ensure geometry column exists.
if "geometry_json" not in hex_df.columns:
    hex_df["geometry_json"] = None

missing_geometries = hex_df["geometry_json"].isna() | (hex_df["geometry_json"].astype(str).str.strip() == "")
print("Rows missing geometry before H3 fill:", int(missing_geometries.sum()))

for idx in hex_df[missing_geometries].index:
    hex_df.at[idx, "geometry_json"] = get_h3_geometry_json(hex_df.at[idx, "region_id"])

# Area in degrees squared is optional and not a true square-meter area.
if "area_degrees_sq" not in hex_df.columns:
    hex_df["area_degrees_sq"] = np.nan

# Build RAG text based on the updated labels.
hex_df["rag_text"] = hex_df.apply(
    lambda row: (
        f"H3 region {row.region_id} is assigned to DBSCAN cluster {int(row.cluster_id)}. "
        f"Its centroid is latitude {float(row.centroid_lat):.6f}, longitude {float(row.centroid_lon):.6f}. "
        f"DBSCAN cluster -1 means this cell is noise or an outlier. "
        f"Non-negative cluster IDs indicate grouped cells found by DBSCAN."
    ),
    axis=1
)

# Final column order.
hex_df = hex_df[[
    "feature_id", "region_id", "cluster_id", "color", "centroid_lat", "centroid_lon",
    "area_degrees_sq", "geometry_json", "rag_text"
]].copy()

print("Final HexCell table shape:", hex_df.shape)
display(hex_df.head())

Merged geometry/centroid fields from hex_cells_from_DBScan_K1_R10.csv.
Rows missing centroid before H3 fill: 20
Rows missing geometry before H3 fill: 20
Final HexCell table shape: (3836, 9)


,feature_id,region_id,cluster_id,color,centroid_lat,centroid_lon,area_degrees_sq,geometry_json,rag_text
0,3761,8a2a8a892717fff,0,#1f77b4,37.207991,-80.406175,0.000001,"{""coordinates"":[[[-80.40599576255426,37.208609...",H3 region 8a2a8a892717fff is assigned to DBSCA...
1,2674,8a2a8ad4a0d7fff,1,#ff7f0e,37.259709,-80.425079,0.000001,"{""coordinates"":[[[-80.42490022629997,37.260329...",H3 region 8a2a8ad4a0d7fff is assigned to DBSCA...
2,1692,8a2a8ad5931ffff,2,#2ca02c,37.254338,-80.445869,0.000001,"{""coordinates"":[[[-80.44569051098459,37.254957...",H3 region 8a2a8ad5931ffff is assigned to DBSCA...
3,98,8a2a8ad4ad87fff,3,#d62728,37.271821,-80.427740,0.000001,"{""coordinates"":[[[-80.4275609667186,37.2724404...",H3 region 8a2a8ad4ad87fff is assigned to DBSCA...
4,55,8a2a8ac66557fff,4,#9467bd,37.243389,-80.468259,0.000001,"{""coordinates"":[[[-80.46808070846453,37.244008...",H3 region 8a2a8ac66557fff is assigned to DBSCA...


## 5. Build the cluster summary table

Each row becomes a `(:Cluster)` node in Neo4j.

In [ ]:
#cluster dataframe population
cluster_summary_df = (
    hex_df
    .groupby("cluster_id")
    .agg(
        cell_count=("region_id", "count"),
        centroid_lat=("centroid_lat", "mean"),
        centroid_lon=("centroid_lon", "mean")
    )
    .reset_index()
    .sort_values("cluster_id")
)

#adjusting labels and filters to data in dataframe
cluster_summary_df["label"] = cluster_summary_df["cluster_id"].apply(
    lambda x: "Noise / Outliers" if int(x) == -1 else f"Cluster {int(x)}"
)
cluster_summary_df["color"] = cluster_summary_df["cluster_id"].apply(cluster_color)
cluster_summary_df["cluster_uid"] = cluster_summary_df["cluster_id"].apply(
    lambda x: f"{RUN_NAME}:{int(x)}"
)
cluster_summary_df["run_name"] = RUN_NAME
cluster_summary_df["rag_text"] = cluster_summary_df.apply(
    lambda row: (
        f"{row.label} in run {RUN_NAME} contains {int(row.cell_count)} H3 cells. "
        f"The approximate cluster centroid is latitude {float(row.centroid_lat):.6f}, "
        f"longitude {float(row.centroid_lon):.6f}. "
        f"Cluster -1 represents DBSCAN noise/outlier cells."
    ),
    axis=1
)

print("Cluster summary shape:", cluster_summary_df.shape)
display(cluster_summary_df)

Cluster summary shape: (38, 9)


,cluster_id,cell_count,centroid_lat,centroid_lon,label,color,cluster_uid,run_name,rag_text
0,-1,818,37.228386,-80.427092,Noise / Outliers,#808080,DBSCAN_Blacksburg_VA_updated:-1,DBSCAN_Blacksburg_VA_updated,Noise / Outliers in run DBSCAN_Blacksburg_VA_u...
1,0,301,37.199482,-80.404079,Cluster 0,#1f77b4,DBSCAN_Blacksburg_VA_updated:0,DBSCAN_Blacksburg_VA_updated,Cluster 0 in run DBSCAN_Blacksburg_VA_updated ...
2,1,5,37.259331,-80.428729,Cluster 1,#ff7f0e,DBSCAN_Blacksburg_VA_updated:1,DBSCAN_Blacksburg_VA_updated,Cluster 1 in run DBSCAN_Blacksburg_VA_updated ...
3,2,20,37.256803,-80.439192,Cluster 2,#2ca02c,DBSCAN_Blacksburg_VA_updated:2,DBSCAN_Blacksburg_VA_updated,Cluster 2 in run DBSCAN_Blacksburg_VA_updated ...
4,3,559,37.236730,-80.425834,Cluster 3,#d62728,DBSCAN_Blacksburg_VA_updated:3,DBSCAN_Blacksburg_VA_updated,Cluster 3 in run DBSCAN_Blacksburg_VA_updated ...
5,4,345,37.237703,-80.449411,Cluster 4,#9467bd,DBSCAN_Blacksburg_VA_updated:4,DBSCAN_Blacksburg_VA_updated,Cluster 4 in run DBSCAN_Blacksburg_VA_updated ...
6,5,362,37.238785,-80.445793,Cluster 5,#8c564b,DBSCAN_Blacksburg_VA_updated:5,DBSCAN_Blacksburg_VA_updated,Cluster 5 in run DBSCAN_Blacksburg_VA_updated ...
7,6,56,37.216280,-80.438808,Cluster 6,#e377c2,DBSCAN_Blacksburg_VA_updated:6,DBSCAN_Blacksburg_VA_updated,Cluster 6 in run DBSCAN_Blacksburg_VA_updated ...
8,7,474,37.235236,-80.410737,Cluster 7,#7f7f7f,DBSCAN_Blacksburg_VA_updated:7,DBSCAN_Blacksburg_VA_updated,Cluster 7 in run DBSCAN_Blacksburg_VA_updated ...
9,8,60,37.256871,-80.421794,Cluster 8,#bcbd22,DBSCAN_Blacksburg_VA_updated:8,DBSCAN_Blacksburg_VA_updated,Cluster 8 in run DBSCAN_Blacksburg_VA_updated ...


## 6. Save cleaned import files

These are useful for backup, inspection, or later import.

In [7]:
hex_out = OUTPUT_DIR / "updated_hex_cells_for_neo4j.csv"
cluster_out = OUTPUT_DIR / "updated_clusters_for_neo4j.csv"

hex_df.to_csv(hex_out, index=False)
cluster_summary_df.to_csv(cluster_out, index=False)

print("Wrote:")
print(hex_out.resolve())
print(cluster_out.resolve())

Wrote:
C:\Users\tdkim\Downloads\ClusteringOutputRag-20260505T223622Z-3-001\ClusteringOutputRag\neo4j_updated_cluster_outputs\updated_hex_cells_for_neo4j.csv
C:\Users\tdkim\Downloads\ClusteringOutputRag-20260505T223622Z-3-001\ClusteringOutputRag\neo4j_updated_cluster_outputs\updated_clusters_for_neo4j.csv


## 7. Optional: render the updated cluster map in Jupyter

This creates a Folium map from the **updated cluster labels** and saves it as HTML. Operates as generally just a visual.

In [ ]:
import folium
from IPython.display import display

center_lat = float(hex_df["centroid_lat"].mean())
center_lon = float(hex_df["centroid_lon"].mean())

fmap = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles="CartoDB Voyager")

#For each cluster, plots it in a geoJSON format for a html file
for cluster_id in sorted(hex_df["cluster_id"].unique()):
    subset = hex_df[hex_df["cluster_id"] == cluster_id]

    features = []
    for row in subset.itertuples(index=False):
        features.append({
            "type": "Feature",
            "geometry": json.loads(row.geometry_json),
            "properties": {
                "region_id": row.region_id,
                "cluster_id": int(row.cluster_id),
                "color": row.color,
                "centroid_lat": float(row.centroid_lat),
                "centroid_lon": float(row.centroid_lon),
            }
        })

    feature_collection = {"type": "FeatureCollection", "features": features}
    layer_name = "Noise (-1)" if int(cluster_id) == -1 else f"Cluster {int(cluster_id)}"

    folium.GeoJson(
        feature_collection,
        name=layer_name,
        style_function=lambda feature: {
            "fillColor": feature["properties"]["color"],
            "color": "black",
            "weight": 0.35,
            "fillOpacity": 0.75,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["region_id", "cluster_id"],
            aliases=["H3 region:", "Cluster:"],
        )
    ).add_to(fmap)

folium.LayerControl().add_to(fmap)

map_out = OUTPUT_DIR / "updated_dbscan_Blacksburg_VA_clusters.html"
fmap.save(map_out)
print("Saved map to:", map_out.resolve())

display(fmap)

Saved map to: C:\Users\tdkim\Downloads\ClusteringOutputRag-20260505T223622Z-3-001\ClusteringOutputRag\neo4j_updated_cluster_outputs\updated_dbscan_Blacksburg_VA_clusters.html


# Neo4j Import

Start Neo4j Desktop before running the next cells. Make sure the database is running and that you know the password for the `neo4j` user.
To download the Neo4j desktop app, go to: https://neo4j.com/download/?
Create a database and if you make any login/general information differently, adjust it in the cell below

## 8. Connect to Neo4j
MAKE SURE that your Connection URI in your Neo4J desktop app matches the code below. Won't work if it doesn't match.

In [9]:
from neo4j import GraphDatabase
from getpass import getpass

NEO4J_URI = "neo4j://127.0.0.1:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "password123"
NEO4J_DATABASE = "neo4j"

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD)
)

driver.verify_connectivity()
print("Connected to Neo4j")

Connected to Neo4j


## 9. ONLY RUN THE CELL BELOW IF IT HASN'T ALREADY BEEN RAN FOR A PARTICULAR NEO4J RUN: 

This deletes the `DBSCANRun` node and `Cluster` nodes for `RUN_NAME`. It keeps the `HexCell` nodes, because those can be reused across runs.

This was made to just adjust previous data from a run that used less-than-optimal clustering.

Again, do NOT run it again for the same run within Neo4j. It will just throw errors because the nodes have already been "cleaned" out.

In [10]:
OVERWRITE_THIS_RUN = True

if OVERWRITE_THIS_RUN:
    driver.execute_query(
        """
        MATCH (r:DBSCANRun {name: $run_name})
        DETACH DELETE r
        """,
        run_name=RUN_NAME,
        database_=NEO4J_DATABASE
    )

    driver.execute_query(
        """
        MATCH (c:Cluster {run_name: $run_name})
        DETACH DELETE c
        """,
        run_name=RUN_NAME,
        database_=NEO4J_DATABASE
    )

    driver.execute_query(
        """
        MATCH (h:HexCell)-[a:ASSIGNED_TO {run_name: $run_name}]->()
        DELETE a
        """,
        run_name=RUN_NAME,
        database_=NEO4J_DATABASE
    )

    print(f"Cleared previous graph objects for run: {RUN_NAME}")
else:
    print("Keeping existing graph objects.")

Cleared previous graph objects for run: DBSCAN_Blacksburg_VA_updated


## 10. Create constraints and indexes

This uses a single `cluster_uid` property instead of a composite constraint, which is more robust across Neo4j setups.

In [17]:
constraints_and_indexes = [
    """
    CREATE CONSTRAINT dbscan_run_name IF NOT EXISTS
    FOR (r:DBSCANRun)
    REQUIRE r.name IS UNIQUE
    """,

    """
    CREATE CONSTRAINT hex_region_id IF NOT EXISTS
    FOR (h:HexCell)
    REQUIRE h.region_id IS UNIQUE
    """,

    """
    CREATE CONSTRAINT cluster_uid IF NOT EXISTS
    FOR (c:Cluster)
    REQUIRE c.cluster_uid IS UNIQUE
    """,

    """
    CREATE POINT INDEX hex_centroid IF NOT EXISTS
    FOR (h:HexCell)
    ON (h.centroid)
    """
]

for query in constraints_and_indexes:
    driver.execute_query(query, database_=NEO4J_DATABASE)

print("Constraints and indexes created.")

Constraints and indexes created.


## 11. Load DBSCAN run, cluster nodes, hex cell nodes, and relationships

In [ ]:
# This loads in batches of a certain size
def batches(records, size=500):
    for i in range(0, len(records), size):
        yield records[i:i + size]

# Cleans clusters in dataframe
def clean_records(df):
    """Convert DataFrame rows to Neo4j-friendly dictionaries."""
    return df.replace({np.nan: None}).to_dict("records")

hex_records = clean_records(hex_df)
cluster_records = clean_records(cluster_summary_df)

print("Hex records:", len(hex_records))
print("Cluster records:", len(cluster_records))

Hex records: 3836
Cluster records: 38


In [12]:
# Create one DBSCAN run node.
driver.execute_query(
    """
    MERGE (r:DBSCANRun {name: $run_name})
    SET r.source_file = $source_file,
        r.place = $place,
        r.algorithm = $algorithm,
        r.last_loaded_at = datetime(),
        r.total_cells = $total_cells,
        r.total_cluster_labels = $total_cluster_labels,
        r.noise_cells = $noise_cells
    """,
    run_name=RUN_NAME,
    source_file=str(CLUSTER_CSV.name),
    place=PLACE_NAME,
    algorithm=ALGORITHM,
    total_cells=int(len(hex_df)),
    total_cluster_labels=int(hex_df["cluster_id"].nunique()),
    noise_cells=int((hex_df["cluster_id"] == -1).sum()),
    database_=NEO4J_DATABASE
)

print("Created/updated DBSCANRun node.")

Created/updated DBSCANRun node.


In [13]:
# Load Cluster nodes.
driver.execute_query(
    """
    UNWIND $rows AS row

    MATCH (r:DBSCANRun {name: $run_name})

    MERGE (c:Cluster {cluster_uid: row.cluster_uid})
    SET c.run_name = $run_name,
        c.cluster_id = row.cluster_id,
        c.label = row.label,
        c.color = row.color,
        c.cell_count = row.cell_count,
        c.centroid_lat = row.centroid_lat,
        c.centroid_lon = row.centroid_lon,
        c.centroid = point({longitude: row.centroid_lon, latitude: row.centroid_lat}),
        c.rag_text = row.rag_text,
        c.updated_at = datetime()

    MERGE (r)-[:HAS_CLUSTER]->(c)
    """,
    rows=cluster_records,
    run_name=RUN_NAME,
    database_=NEO4J_DATABASE
)

print(f"Loaded {len(cluster_records)} Cluster nodes.")

Loaded 38 Cluster nodes.


In [14]:
# Load HexCell nodes and assignment relationships.
for batch_num, batch in enumerate(batches(hex_records, size=500), start=1):
    driver.execute_query(
        """
        UNWIND $rows AS row

        MATCH (r:DBSCANRun {name: $run_name})
        MATCH (c:Cluster {cluster_uid: $run_name + ':' + toString(row.cluster_id)})

        MERGE (h:HexCell {region_id: row.region_id})
        SET h.feature_id = row.feature_id,
            h.current_cluster_id = row.cluster_id,
            h.color = row.color,
            h.centroid_lat = row.centroid_lat,
            h.centroid_lon = row.centroid_lon,
            h.centroid = point({longitude: row.centroid_lon, latitude: row.centroid_lat}),
            h.area_degrees_sq = row.area_degrees_sq,
            h.geometry_json = row.geometry_json,
            h.rag_text = row.rag_text,
            h.updated_at = datetime()

        MERGE (r)-[:HAS_CELL]->(h)
        MERGE (c)-[:HAS_CELL]->(h)
        MERGE (h)-[a:ASSIGNED_TO {run_name: $run_name}]->(c)
        SET a.cluster_id = row.cluster_id,
            a.updated_at = datetime()
        """,
        rows=batch,
        run_name=RUN_NAME,
        database_=NEO4J_DATABASE
    )
    print(f"Loaded batch {batch_num} with {len(batch)} HexCell rows.")

print("Finished loading updated clusters into Neo4j.")

Loaded batch 1 with 500 HexCell rows.
Loaded batch 2 with 500 HexCell rows.
Loaded batch 3 with 500 HexCell rows.
Loaded batch 4 with 500 HexCell rows.
Loaded batch 5 with 500 HexCell rows.
Loaded batch 6 with 500 HexCell rows.
Loaded batch 7 with 500 HexCell rows.
Loaded batch 8 with 336 HexCell rows.
Finished loading updated clusters into Neo4j.


## 12. Verify the Neo4j import

In [15]:
result = driver.execute_query(
    """
    MATCH (r:DBSCANRun {name: $run_name})
    OPTIONAL MATCH (r)-[:HAS_CLUSTER]->(c:Cluster)
    OPTIONAL MATCH (r)-[:HAS_CELL]->(h:HexCell)
    RETURN
        r.name AS run_name,
        r.total_cells AS total_cells_property,
        r.total_cluster_labels AS total_cluster_labels,
        r.noise_cells AS noise_cells,
        count(DISTINCT c) AS cluster_nodes,
        count(DISTINCT h) AS hex_nodes
    """,
    run_name=RUN_NAME,
    database_=NEO4J_DATABASE
)

verification_df = pd.DataFrame([dict(record) for record in result.records])
display(verification_df)

,run_name,total_cells_property,total_cluster_labels,noise_cells,cluster_nodes,hex_nodes
0,DBSCAN_Blacksburg_VA_updated,3836,38,818,38,3836


In [16]:
result = driver.execute_query(
    """
    MATCH (:DBSCANRun {name: $run_name})-[:HAS_CLUSTER]->(c:Cluster)
    RETURN
        c.cluster_id AS cluster_id,
        c.label AS label,
        c.cell_count AS cell_count,
        c.centroid_lat AS centroid_lat,
        c.centroid_lon AS centroid_lon
    ORDER BY c.cluster_id
    """,
    run_name=RUN_NAME,
    database_=NEO4J_DATABASE
)

cluster_check_df = pd.DataFrame([dict(record) for record in result.records])
display(cluster_check_df)

,cluster_id,label,cell_count,centroid_lat,centroid_lon
0,-1,Noise / Outliers,818,37.228386,-80.427092
1,0,Cluster 0,301,37.199482,-80.404079
2,1,Cluster 1,5,37.259331,-80.428729
3,2,Cluster 2,20,37.256803,-80.439192
4,3,Cluster 3,559,37.236730,-80.425834
5,4,Cluster 4,345,37.237703,-80.449411
6,5,Cluster 5,362,37.238785,-80.445793
7,6,Cluster 6,56,37.216280,-80.438808
8,7,Cluster 7,474,37.235236,-80.410737
9,8,Cluster 8,60,37.256871,-80.421794


## 13. Useful graph query functions

These put some consistent form to the RAG output

In [ ]:
# Query structure for summary from driver
def query_cluster_summary():
    result = driver.execute_query(
        """
        MATCH (:DBSCANRun {name: $run_name})-[:HAS_CLUSTER]->(c:Cluster)
        RETURN
            c.cluster_id AS cluster_id,
            c.label AS label,
            c.cell_count AS cell_count,
            c.centroid_lat AS centroid_lat,
            c.centroid_lon AS centroid_lon
        ORDER BY c.cell_count DESC
        """,
        run_name=RUN_NAME,
        database_=NEO4J_DATABASE
    )
    return pd.DataFrame([dict(record) for record in result.records])

#Query structuring for cluster from driver
def query_cluster(cluster_id, limit=20):
    result = driver.execute_query(
        """
        MATCH (:DBSCANRun {name: $run_name})-[:HAS_CLUSTER]->(c:Cluster {cluster_id: $cluster_id})
        MATCH (c)-[:HAS_CELL]->(h:HexCell)
        RETURN
            c.cluster_id AS cluster_id,
            c.label AS cluster_label,
            c.cell_count AS cluster_cell_count,
            h.region_id AS region_id,
            h.centroid_lat AS lat,
            h.centroid_lon AS lon,
            h.rag_text AS rag_text
        LIMIT $limit
        """,
        run_name=RUN_NAME,
        cluster_id=int(cluster_id),
        limit=int(limit),
        database_=NEO4J_DATABASE
    )
    return pd.DataFrame([dict(record) for record in result.records])

#Query structuring for hex from driver
def query_hex(region_id):
    result = driver.execute_query(
        """
        MATCH (h:HexCell {region_id: $region_id})-[a:ASSIGNED_TO {run_name: $run_name}]->(c:Cluster)
        RETURN
            h.region_id AS region_id,
            a.cluster_id AS cluster_id,
            h.centroid_lat AS lat,
            h.centroid_lon AS lon,
            h.geometry_json AS geometry_json,
            h.rag_text AS hex_text,
            c.label AS cluster_label,
            c.cell_count AS cluster_cell_count,
            c.rag_text AS cluster_text
        """,
        region_id=str(region_id),
        run_name=RUN_NAME,
        database_=NEO4J_DATABASE
    )
    return pd.DataFrame([dict(record) for record in result.records])


def query_noise_points(limit=20):
    return query_cluster(-1, limit=limit)

print("Query functions ready.")

Query functions ready.


In [ ]:
#Checking loaded clusters again
display(query_cluster_summary().head(10))
display(query_noise_points(limit=10))

sample_region = hex_df.iloc[0]["region_id"]
print("Sample region:", sample_region)
display(query_hex(sample_region))

,cluster_id,label,cell_count,centroid_lat,centroid_lon
0,-1,Noise / Outliers,818,37.228386,-80.427092
1,3,Cluster 3,559,37.236730,-80.425834
2,7,Cluster 7,474,37.235236,-80.410737
3,5,Cluster 5,362,37.238785,-80.445793
4,4,Cluster 4,345,37.237703,-80.449411
5,9,Cluster 9,315,37.222471,-80.427048
6,0,Cluster 0,301,37.199482,-80.404079
7,11,Cluster 11,78,37.244320,-80.419895
8,13,Cluster 13,78,37.207560,-80.440316
9,15,Cluster 15,62,37.245079,-80.453377


,cluster_id,cluster_label,cluster_cell_count,region_id,lat,lon,rag_text
0,-1,Noise / Outliers,818,8a2a8a892c17fff,37.218326,-80.403955,H3 region 8a2a8a892c17fff is assigned to DBSCA...
1,-1,Noise / Outliers,818,8a2a8ac6001ffff,37.223982,-80.444621,H3 region 8a2a8ac6001ffff is assigned to DBSCA...
2,-1,Noise / Outliers,818,8a2a8ad4a2cffff,37.251077,-80.419876,H3 region 8a2a8ad4a2cffff is assigned to DBSCA...
3,-1,Noise / Outliers,818,8a2a8ac6050ffff,37.228084,-80.451898,H3 region 8a2a8ac6050ffff is assigned to DBSCA...
4,-1,Noise / Outliers,818,8a2a8ad4a0cffff,37.258421,-80.423142,H3 region 8a2a8ad4a0cffff is assigned to DBSCA...
5,-1,Noise / Outliers,818,8a2a8ac60017fff,37.225030,-80.445088,H3 region 8a2a8ac60017fff is assigned to DBSCA...
6,-1,Noise / Outliers,818,8a2a8ac653b7fff,37.223149,-80.415636,H3 region 8a2a8ac653b7fff is assigned to DBSCA...
7,-1,Noise / Outliers,818,8a2a8ac63d77fff,37.214841,-80.454176,H3 region 8a2a8ac63d77fff is assigned to DBSCA...
8,-1,Noise / Outliers,818,8a2a8ac6525ffff,37.216618,-80.411371,H3 region 8a2a8ac6525ffff is assigned to DBSCA...
9,-1,Noise / Outliers,818,8a2a8ad59217fff,37.252719,-80.447875,H3 region 8a2a8ad59217fff is assigned to DBSCA...


Sample region: 8a2a8a892717fff


,region_id,cluster_id,lat,lon,geometry_json,hex_text,cluster_label,cluster_cell_count,cluster_text
0,8a2a8a892717fff,0,37.207991,-80.406175,"{""coordinates"":[[[-80.40599576255426,37.208609...",H3 region 8a2a8a892717fff is assigned to DBSCA...,Cluster 0,301,Cluster 0 in run DBSCAN_Blacksburg_VA_updated ...


## 14. create H3 neighbor relationships

This is useful for graph traversal, such as finding nearby cells or neighboring clusters.

In [ ]:
CREATE_H3_NEIGHBOR_RELATIONSHIPS = True

#Creates the k ring based on input k (default 1)
def h3_grid_disk(region_id, k=1):
    if hasattr(h3, "grid_disk"):
        return list(h3.grid_disk(region_id, k))
    else:
        return list(h3.k_ring(region_id, k))

#Generates neighbor relationships based on k ring and radius
if CREATE_H3_NEIGHBOR_RELATIONSHIPS:
    all_region_ids = set(hex_df["region_id"].astype(str))
    neighbor_edges = []

    for region_id in all_region_ids:
        for neighbor_id in h3_grid_disk(region_id, k=1):
            neighbor_id = str(neighbor_id)
            if neighbor_id == region_id:
                continue
            if neighbor_id in all_region_ids:
                a, b = sorted([region_id, neighbor_id])
                neighbor_edges.append((a, b))

    neighbor_edges = sorted(set(neighbor_edges))
    neighbor_df = pd.DataFrame(neighbor_edges, columns=["source_region_id", "target_region_id"])

    print("Neighbor edges:", len(neighbor_df))
    display(neighbor_df.head())
else:
    neighbor_df = pd.DataFrame(columns=["source_region_id", "target_region_id"])
    print("Skipped neighbor generation.")

Neighbor edges: 11076


,source_region_id,target_region_id
0,8a2a8a890497fff,8a2a8a89049ffff
1,8a2a8a890497fff,8a2a8a892867fff
2,8a2a8a890497fff,8a2a8a89286ffff
3,8a2a8a890497fff,8a2a8a89294ffff
4,8a2a8a89049ffff,8a2a8a89286ffff


In [ ]:
#Prints neighbor edges by batches of 1000
if CREATE_H3_NEIGHBOR_RELATIONSHIPS and not neighbor_df.empty:
    neighbor_records = neighbor_df.to_dict("records")

    for batch_num, batch in enumerate(batches(neighbor_records, size=1000), start=1):
        driver.execute_query(
            """
            UNWIND $rows AS row

            MATCH (a:HexCell {region_id: row.source_region_id})
            MATCH (b:HexCell {region_id: row.target_region_id})

            MERGE (a)-[:H3_NEIGHBOR_OF]->(b)
            MERGE (b)-[:H3_NEIGHBOR_OF]->(a)
            """,
            rows=batch,
            database_=NEO4J_DATABASE
        )
        print(f"Loaded neighbor batch {batch_num} with {len(batch)} edges.")

    print("Finished loading H3 neighbor relationships.")
else:
    print("No neighbor relationships loaded.")

Loaded neighbor batch 1 with 1000 edges.
Loaded neighbor batch 2 with 1000 edges.
Loaded neighbor batch 3 with 1000 edges.
Loaded neighbor batch 4 with 1000 edges.
Loaded neighbor batch 5 with 1000 edges.
Loaded neighbor batch 6 with 1000 edges.
Loaded neighbor batch 7 with 1000 edges.
Loaded neighbor batch 8 with 1000 edges.
Loaded neighbor batch 9 with 1000 edges.
Loaded neighbor batch 10 with 1000 edges.
Loaded neighbor batch 11 with 1000 edges.
Loaded neighbor batch 12 with 76 edges.
Finished loading H3 neighbor relationships.


# Vector RAG Setup

The current `rag_text` mainly describes H3 region IDs, cluster labels, and coordinates. To get richer neighborhood explanations, add OSM/Hex2Vec feature counts such as restaurant, park, supermarket, road, building, and land-use fields to `rag_text` later.

## 15. Create and store text embeddings

In [21]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

EMBEDDING_DIM = len(embedding_model.encode("test sentence").tolist())
print("Embedding dimension:", EMBEDDING_DIM)

Embedding dimension: 384


In [22]:
# This may take a little time.
hex_df["embedding"] = hex_df["rag_text"].apply(lambda text: embedding_model.encode(str(text)).tolist())
print("Embeddings created for", len(hex_df), "hex cells.")

Embeddings created for 3836 hex cells.


In [23]:
embedding_records = hex_df[["region_id", "embedding"]].to_dict("records")

for batch_num, batch in enumerate(batches(embedding_records, size=250), start=1):
    driver.execute_query(
        """
        UNWIND $rows AS row
        MATCH (h:HexCell {region_id: row.region_id})
        SET h.embedding = row.embedding
        """,
        rows=batch,
        database_=NEO4J_DATABASE
    )
    print(f"Loaded embedding batch {batch_num} with {len(batch)} rows.")

print("Finished loading embeddings into Neo4j.")

Loaded embedding batch 1 with 250 rows.
Loaded embedding batch 2 with 250 rows.
Loaded embedding batch 3 with 250 rows.
Loaded embedding batch 4 with 250 rows.
Loaded embedding batch 5 with 250 rows.
Loaded embedding batch 6 with 250 rows.
Loaded embedding batch 7 with 250 rows.
Loaded embedding batch 8 with 250 rows.
Loaded embedding batch 9 with 250 rows.
Loaded embedding batch 10 with 250 rows.
Loaded embedding batch 11 with 250 rows.
Loaded embedding batch 12 with 250 rows.
Loaded embedding batch 13 with 250 rows.
Loaded embedding batch 14 with 250 rows.
Loaded embedding batch 15 with 250 rows.
Loaded embedding batch 16 with 86 rows.
Finished loading embeddings into Neo4j.


In [24]:
driver.execute_query(
    f"""
    CREATE VECTOR INDEX hex_rag_embedding IF NOT EXISTS
    FOR (h:HexCell)
    ON (h.embedding)
    OPTIONS {{
        indexConfig: {{
            `vector.dimensions`: {EMBEDDING_DIM},
            `vector.similarity_function`: 'cosine'
        }}
    }}
    """,
    database_=NEO4J_DATABASE
)

print("Vector index created or already exists.")

Vector index created or already exists.


## 16. Vector search function

In [ ]:
#Checks vector similarities and similarities in database
def vector_search(question, k=8):
    question_embedding = embedding_model.encode(question).tolist()

    result = driver.execute_query(
        """
        CALL db.index.vector.queryNodes('hex_rag_embedding', $k, $embedding)
        YIELD node, score

        OPTIONAL MATCH (node)-[a:ASSIGNED_TO {run_name: $run_name}]->(c:Cluster)

        RETURN
            node.region_id AS region_id,
            a.cluster_id AS cluster_id,
            node.centroid_lat AS lat,
            node.centroid_lon AS lon,
            node.rag_text AS text,
            c.label AS cluster_label,
            c.cell_count AS cluster_cell_count,
            score AS score
        ORDER BY score DESC
        """,
        k=int(k),
        embedding=question_embedding,
        run_name=RUN_NAME,
        database_=NEO4J_DATABASE
    )

    return pd.DataFrame([dict(record) for record in result.records])

#Actual question being asked. Adjust this for a new question.
display(vector_search("Which cells are DBSCAN noise or outliers?", k=10))

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=9, offset=9>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 2, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        CALL db.index.vector.queryNodes('hex_rag_embedding', $k, $embedding)\n        YIELD node, score\n\n        OPTIONAL MATCH (node)-[a:ASSIGNED_TO {run_name: $run_name}]->(c:Cluster)\n\n        RETURN\n            node.region_id AS region_id,\n            a.cluster_id AS cluster_id,\n            node.centroid_lat AS lat,\n            

,region_id,cluster_id,lat,lon,text,cluster_label,cluster_cell_count,score
0,8a2a8ac6450ffff,5,37.246757,-80.441130,H3 region 8a2a8ac6450ffff is assigned to DBSCA...,Cluster 5,362,0.798144
1,8a2a8ac64c2ffff,5,37.250565,-80.434645,H3 region 8a2a8ac64c2ffff is assigned to DBSCA...,Cluster 5,362,0.798030
2,8a2a8ad59af7fff,5,37.256528,-80.441390,H3 region 8a2a8ad59af7fff is assigned to DBSCA...,Cluster 5,362,0.798018
3,8a2a8ac64d9ffff,5,37.252332,-80.439522,H3 region 8a2a8ac64d9ffff is assigned to DBSCA...,Cluster 5,362,0.797505
4,8a2a8ac6144ffff,5,37.206729,-80.434218,H3 region 8a2a8ac6144ffff is assigned to DBSCA...,Cluster 5,362,0.797352
5,8a2a8ac6676ffff,5,37.235570,-80.462048,H3 region 8a2a8ac6676ffff is assigned to DBSCA...,Cluster 5,362,0.797241
6,8a2a8ac667b7fff,5,37.238862,-80.470331,H3 region 8a2a8ac667b7fff is assigned to DBSCA...,Cluster 5,362,0.797161
7,8a2a8ac64d77fff,5,37.252184,-80.432639,H3 region 8a2a8ac64d77fff is assigned to DBSCA...,Cluster 5,362,0.797124
8,8a2a8ac64d0ffff,5,37.252424,-80.434109,H3 region 8a2a8ac64d0ffff is assigned to DBSCA...,Cluster 5,362,0.797083
9,8a2a8ac6692ffff,5,37.249241,-80.450416,H3 region 8a2a8ac6692ffff is assigned to DBSCA...,Cluster 5,362,0.796995


## 18. Qwen/Ollama RAG

Install Ollama from `https://ollama.com/download/windows`, then run in PowerShell:

```powershell
ollama pull qwen2.5:7b
```

If you use a different model, change `QWEN_MODEL` below.

In [ ]:
# This cell assumes vector_search(question, k=...) already exists and is ran.

import requests

try:
    import ollama
except ImportError:
    ollama = None


QWEN_MODEL = "qwen2.5:7b"   # Change this if you pulled a different model, e.g. "qwen2.5:3b"

#Checks ollama to ensure the model is running or throws issues (terminal messages) if something is missing
def check_ollama():
    """
    Checks whether Ollama is installed, running, and reachable at localhost:11434.
    """
    if ollama is None:
        print("The Python package 'ollama' is not installed.")
        print("Run this in a notebook cell:")
        print("%pip install ollama")
        return False

    try:
        response = requests.get("http://localhost:11434/api/tags", timeout=5)
        response.raise_for_status()

        models = response.json().get("models", [])
        model_names = [m.get("name") for m in models]

        print("Ollama is running.")
        print("Installed models:", model_names)

        if QWEN_MODEL not in model_names:
            print(f"WARNING: {QWEN_MODEL} is not installed.")
            print(f"Run this in PowerShell:")
            print(f"ollama pull {QWEN_MODEL}")

        return True

    except Exception as e:
        print("Ollama is not reachable from Python.")
        print("To use Qwen, install/start Ollama and run:")
        print("ollama serve")
        print()
        print("Then test with:")
        print("curl http://localhost:11434/api/tags")
        print()
        print("Original error:")
        print(e)
        return False

#Sends the actual question to the Qwen LLM
def ask_qwen_graph_rag(question, k=8):
    """
    Retrieves cluster/H3 context from Neo4j using vector_search(),
    then sends that context to Qwen through Ollama.
    """

    hits = vector_search(question, k=k)

    if hits is None or hits.empty:
        return "No relevant Neo4j context was retrieved."

    context = "\n".join(
        f"- region_id={row.region_id}, cluster_id={row.cluster_id}, "
        f"cluster_label={row.cluster_label}, "
        f"lat={row.lat:.6f}, lon={row.lon:.6f}, "
        f"score={row.score:.3f}: {row.text}"
        for row in hits.itertuples(index=False)
    )
#This is the actual prompt itself. It was structured to return information in the same format as the RAG based on City
#Structure for response was already defined earlier but this adds it into the worded prompt itself.
    prompt = f"""
You are a geospatial RAG assistant analyzing DBSCAN clusters stored in Neo4j.

Use only the retrieved context. Do not invent OSM features such as restaurants, parks, shops, buildings, roads, land use, or POI counts unless they appear in the context.

Return your answer in this format:

--- INSIGHTS ---

### Direct Answer
Answer the question clearly.

### Relevant H3 Regions
List the most relevant H3 region IDs, their DBSCAN cluster IDs, and coordinates.

### Cluster Interpretation
Explain what the retrieved cluster IDs mean.
Remember: cluster -1 means DBSCAN noise/outlier.

### Spatial Pattern
Explain any spatial pattern that can be inferred from the coordinates and cluster assignments.

### Limitations
Explain what extra data would be needed for richer neighborhood analysis.

Question:
{question}

Retrieved context:
{context}
""".strip()

    if not check_ollama():
        print()
        print("Retrieved context was:")
        print(context)
        return "Ollama/Qwen is not running, but Neo4j retrieval worked."

    response = ollama.chat(
        model=QWEN_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]


print("Qwen/Ollama RAG function ready.")

Qwen/Ollama RAG function ready.


In [ ]:
#Prints the result from the LLM
print(
    ask_qwen_graph_rag(
        "Summarize the DBSCAN noise points and explain what cluster they belong to.",
        k=10
    )
)

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=9, offset=9>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 2, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        CALL db.index.vector.queryNodes('hex_rag_embedding', $k, $embedding)\n        YIELD node, score\n\n        OPTIONAL MATCH (node)-[a:ASSIGNED_TO {run_name: $run_name}]->(c:Cluster)\n\n        RETURN\n            node.region_id AS region_id,\n            a.cluster_id AS cluster_id,\n            node.centroid_lat AS lat,\n            

Ollama is running.
Installed models: ['qwen2.5:7b', 'mistral:latest', 'llama3:latest']
---

### INSIGHTS

#### Direct Answer
There are no DBSCAN noise points in the retrieved context, as all regions have non-negative cluster IDs.

#### Relevant H3 Regions
- **H3 region 8a2a8ad4a8cffff** (cluster_id=5), coordinates: 37.264088, -80.416117
- **H3 region 8a2a8ac610cffff** (cluster_id=5), coordinates: 37.204724, -80.427877
- **H3 region 8a2a8ac6676ffff** (cluster_id=5), coordinates: 37.235570, -80.462048
- **H3 region 8a2a8ac667a7fff** (cluster_id=5), coordinates: 37.238623, -80.468861
- **H3 region 8a2a8ac663affff** (cluster_id=5), coordinates: 37.235902, -80.458107
- **H3 region 8a2a8ac66717fff** (cluster_id=5), coordinates: 37.238384, -80.467392
- **H3 region 8a2a8ac64c07fff** (cluster_id=5), coordinates: 37.250804, -80.436115
- **H3 region 8a2a8ac662b7fff** (cluster_id=5), coordinates: 37.234522, -80.461581
- **H3 region 8a2a8ac64d9ffff** (cluster_id=5), coordinates: 37.252332, -80.4395

## 19. Close the Neo4j driver when finished

In [36]:
# Run this only when you are done using Neo4j in this notebook.
# driver.close()
# print("Neo4j driver closed.")